# 02 - Preprocesamiento y particiones: ISIC 2019

**Propósito del notebook:** Construir la partición estratificada train/validation/test sobre el inventario ya validado de ISIC 2019 (`01_datos_isic2019.ipynb`), verificando primero si es posible agrupar por lesión/paciente para prevenir fuga de datos, y persistir los splits resultantes como artefacto reproducible para los notebooks de modelado.

**Entrada de datos:** este notebook **no vuelve a cargar ni validar** `ISIC_2019_Training_GroundTruth.csv` desde cero. Usa como fuente única `reports/inventory/isic2019_inventario_ocho_clases.csv`, generado y verificado en `01_datos_isic2019.ipynb` (25.331 imágenes, ocho clases, sin duplicados, sin faltantes, casos OOD ya excluidos).

**Orden de trabajo de este notebook:**

1. Rutas portables (reutilizando el mismo patrón que en el notebook 01).
2. Carga del inventario ya validado.
3. Carga y verificación de `ISIC_2019_Training_Metadata.csv`, en particular la columna `lesion_id`, para determinar si es posible agrupar por lesión antes de particionar.
4. Definición de la estrategia de partición (agrupada por lesión si los metadatos lo permiten; documentando la limitación si no).
5. Partición estratificada 70% train / 15% validation / 15% test, semilla 42.
6. Verificación de ausencia de fuga (ninguna lesión ni imagen duplicada entre subconjuntos).
7. Persistencia de los splits en `reports/`.

**Explícitamente fuera de alcance:** cálculo de pesos de clase, normalización/augmentation de imágenes, y entrenamiento. Esos pasos corresponden a los notebooks de construcción de modelos (`03`/`04`) y entrenamiento (`05`).

## Rutas portables del proyecto

Se reutiliza el mismo mecanismo de localizacion de la raiz del proyecto usado en `01_datos_isic2019.ipynb`: buscar hacia arriba desde el directorio de trabajo hasta encontrar `requirements.txt`. A partir de esa raiz se definen las rutas de entrada (inventario del notebook 01, metadatos de ISIC 2019) y de salida (particiones persistidas) de este notebook.

In [1]:
from pathlib import Path


def find_project_root(marker: str = "requirements.txt") -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"No se encontro '{marker}' en ningun directorio superior a {current}. "
        "Ejecuta este notebook desde dentro del repositorio del proyecto."
    )


PROJECT_ROOT = find_project_root()

# Entradas: producidas por 01_datos_isic2019.ipynb y por el dataset descargado
DATA_DIR = PROJECT_ROOT / "data" / "isic2019"
METADATA_CSV = DATA_DIR / "ISIC_2019_Training_Metadata.csv"
INVENTARIO_CSV = PROJECT_ROOT / "reports" / "inventory" / "isic2019_inventario_ocho_clases.csv"

# Salidas de este notebook
REPORTS_DIR = PROJECT_ROOT / "reports"
SPLITS_DIR = REPORTS_DIR / "splits" / "isic2019"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raiz del proyecto              : {PROJECT_ROOT}")
print(f"CSV de metadatos                : {METADATA_CSV}")
print(f"Inventario de entrada (notebook 01): {INVENTARIO_CSV}")
print(f"Directorio de salida de splits   : {SPLITS_DIR}")

Raiz del proyecto              : /home/milat/proyecto_de_titulo
CSV de metadatos                : /home/milat/proyecto_de_titulo/data/isic2019/ISIC_2019_Training_Metadata.csv
Inventario de entrada (notebook 01): /home/milat/proyecto_de_titulo/reports/inventory/isic2019_inventario_ocho_clases.csv
Directorio de salida de splits   : /home/milat/proyecto_de_titulo/reports/splits/isic2019


## Carga de metadatos e inventario, verificacion de `lesion_id`

Se cargan el inventario ya validado del notebook 01 y el archivo oficial de metadatos de ISIC 2019, y se cruzan por `image` para determinar si la columna `lesion_id` esta disponible y que cobertura tiene. Esto define si la particion puede agruparse por lesion (para evitar que fotos de la misma lesion queden repartidas entre train/validation/test) o si esa agrupacion no es posible con los metadatos disponibles.

In [2]:
import pandas as pd

df_inventario = pd.read_csv(INVENTARIO_CSV)
df_metadata = pd.read_csv(METADATA_CSV)

print(f"Filas en inventario (01): {len(df_inventario)}")
print(f"Filas en metadatos      : {len(df_metadata)}")
print(f"Columnas de metadatos   : {list(df_metadata.columns)}")

tiene_lesion_id = "lesion_id" in df_metadata.columns
print(f"\nColumna 'lesion_id' presente: {tiene_lesion_id}")

if tiene_lesion_id:
    n_con_lesion = df_metadata["lesion_id"].notna().sum()
    n_sin_lesion = df_metadata["lesion_id"].isna().sum()
    n_lesiones_unicas = df_metadata["lesion_id"].nunique()
    print(f"Imagenes con lesion_id      : {n_con_lesion}")
    print(f"Imagenes sin lesion_id (NaN): {n_sin_lesion}")
    print(f"Lesiones unicas             : {n_lesiones_unicas}")

Filas en inventario (01): 25331
Filas en metadatos      : 25331
Columnas de metadatos   : ['image', 'age_approx', 'anatom_site_general', 'lesion_id', 'sex']

Columna 'lesion_id' presente: True
Imagenes con lesion_id      : 23247
Imagenes sin lesion_id (NaN): 2084
Lesiones unicas             : 11847


## Verificacion de consistencia de clase dentro de cada lesion

Antes de agrupar por `lesion_id` para la particion, se verifica que todas las imagenes de una misma lesion compartan la misma clase diagnostica. Si una lesion tuviera imagenes con clases distintas, agrupar por lesion no seria valido sin antes documentar y resolver esa inconsistencia.

In [3]:
df_combinado = df_inventario.merge(df_metadata[["image", "lesion_id"]], on="image", how="left")

assert len(df_combinado) == len(df_inventario), "El cruce con metadatos no debe alterar el numero de filas."

con_lesion = df_combinado[df_combinado["lesion_id"].notna()]

clases_por_lesion = con_lesion.groupby("lesion_id")["class"].nunique()
lesiones_inconsistentes = clases_por_lesion[clases_por_lesion > 1]

print(f"Lesiones con mas de una clase diagnostica: {len(lesiones_inconsistentes)}")
if len(lesiones_inconsistentes) > 0:
    print("IDs de lesiones inconsistentes (primeras 10):")
    print(lesiones_inconsistentes.head(10))
else:
    print("Todas las lesiones con lesion_id tienen una clase diagnostica consistente.")

Lesiones con mas de una clase diagnostica: 0
Todas las lesiones con lesion_id tienen una clase diagnostica consistente.


## Definicion del identificador de grupo para la particion

Se define `group_id`: el `lesion_id` cuando esta disponible, o el propio `image` cuando no lo esta (tratando esa imagen como grupo unico de una sola lesion). Esto permite particionar garantizando que ninguna lesion quede repartida entre train, validation y test, incluyendo el caso de imagenes sin `lesion_id` registrado.

La particion se realizara a nivel de grupo (no a nivel de imagen), asignando a cada grupo la clase diagnostica de sus imagenes (ya verificada como consistente), y estratificando por esa clase.

In [4]:
ORDEN_CLASES = ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]

df_combinado["group_id"] = df_combinado["lesion_id"].fillna(df_combinado["image"])

n_grupos = df_combinado["group_id"].nunique()
print(f"Total de imagenes : {len(df_combinado)}")
print(f"Total de grupos (lesiones + imagenes sin lesion_id): {n_grupos}")

df_grupos = (
    df_combinado.groupby("group_id")
    .agg(class_=("class", "first"), n_imagenes=("image", "count"))
    .reset_index()
    .rename(columns={"class_": "class"})
)

print(f"\nFilas en tabla de grupos: {len(df_grupos)}")
display(df_grupos.head())

print("\nDistribucion de tamano de grupo (numero de imagenes por grupo):")
print(df_grupos["n_imagenes"].value_counts().sort_index())

print("\nConteo de grupos por clase:")
print(df_grupos["class"].value_counts().reindex(ORDEN_CLASES))

Total de imagenes : 25331
Total de grupos (lesiones + imagenes sin lesion_id): 13931

Filas en tabla de grupos: 13931


,group_id,class,n_imagenes
0,BCN_0000001,AK,3
1,BCN_0000002,NV,3
2,BCN_0000003,SCC,2
3,BCN_0000004,BCC,6
4,BCN_0000008,BCC,3



Distribucion de tamano de grupo (numero de imagenes por grupo):
n_imagenes
1     8872
2     2505
3     1306
4      468
5      248
6      198
7      112
8       59
9       54
10      24
11      18
12      19
13      10
14       7
15       3
16       3
17       5
18       3
19       6
20       6
21       1
24       1
26       1
27       1
31       1
Name: count, dtype: int64

Conteo de grupos por clase:
class
AK       324
BCC     1310
BKL     1467
DF       113
MEL     1676
NV      8644
SCC      262
VASC     135
Name: count, dtype: int64


## Particion estratificada 70/15/15 a nivel de grupo (semilla 42)

Se particiona `df_grupos` (un registro por lesion/imagen-huerfana) en train/validation/test con proporciones 70/15/15, estratificando por clase diagnostica y usando semilla fija 42, tal como especifica `configs/pilot_m1_m4.yaml`. Como la particion ocurre a nivel de grupo, se garantiza que todas las imagenes de una misma lesion caen en el mismo subconjunto.

Las proporciones finales a nivel de imagen pueden diferir levemente de 70/15/15 exacto, porque los grupos tienen distinto numero de imagenes (entre 1 y mas de 20). Esa diferencia se cuantifica explicitamente en el siguiente paso, no se oculta.

In [5]:
from sklearn.model_selection import train_test_split

SEMILLA_PARTICION = 42

grupos_train, grupos_temp = train_test_split(
    df_grupos,
    test_size=0.30,
    stratify=df_grupos["class"],
    random_state=SEMILLA_PARTICION,
)

grupos_val, grupos_test = train_test_split(
    grupos_temp,
    test_size=0.50,
    stratify=grupos_temp["class"],
    random_state=SEMILLA_PARTICION,
)

print(f"Grupos train      : {len(grupos_train)}")
print(f"Grupos validation : {len(grupos_val)}")
print(f"Grupos test       : {len(grupos_test)}")
print(f"Total             : {len(grupos_train) + len(grupos_val) + len(grupos_test)} (esperado {len(df_grupos)})")

mapa_split = {}
for group_id in grupos_train["group_id"]:
    mapa_split[group_id] = "train"
for group_id in grupos_val["group_id"]:
    mapa_split[group_id] = "validation"
for group_id in grupos_test["group_id"]:
    mapa_split[group_id] = "test"

df_combinado["split"] = df_combinado["group_id"].map(mapa_split)

print("\nDistribucion de imagenes por split:")
conteo_split = df_combinado["split"].value_counts()
print(conteo_split)
print("\nProporcion de imagenes por split:")
print((100 * conteo_split / len(df_combinado)).round(2))

Grupos train      : 9751
Grupos validation : 2090
Grupos test       : 2090
Total             : 13931 (esperado 13931)

Distribucion de imagenes por split:
split
train         17705
validation     3818
test           3808
Name: count, dtype: int64

Proporcion de imagenes por split:
split
train         69.89
validation    15.07
test          15.03
Name: count, dtype: float64


## Verificacion de ausencia de fuga entre splits

Se verifica explicitamente que ningun `group_id` (lesion o imagen sin lesion_id) aparezca en mas de un split, y que la distribucion de clases dentro de cada split se mantenga razonablemente similar a la distribucion global (la estratificacion deberia garantizar esto, pero se comprueba con los datos reales en vez de asumirlo).

In [6]:
grupos_por_split = {
    "train": set(grupos_train["group_id"]),
    "validation": set(grupos_val["group_id"]),
    "test": set(grupos_test["group_id"]),
}

interseccion_train_val = grupos_por_split["train"] & grupos_por_split["validation"]
interseccion_train_test = grupos_por_split["train"] & grupos_por_split["test"]
interseccion_val_test = grupos_por_split["validation"] & grupos_por_split["test"]

print(f"Interseccion train/validation: {len(interseccion_train_val)} grupos")
print(f"Interseccion train/test      : {len(interseccion_train_test)} grupos")
print(f"Interseccion validation/test : {len(interseccion_val_test)} grupos")

sin_fuga = (
    len(interseccion_train_val) == 0
    and len(interseccion_train_test) == 0
    and len(interseccion_val_test) == 0
)
print(f"\nParticion sin fuga de grupos entre splits: {sin_fuga}")

assert df_combinado["split"].isna().sum() == 0, "Hay imagenes sin split asignado."
print("Todas las imagenes tienen un split asignado.")

print("\nDistribucion de clases por split (porcentaje dentro de cada split):")
tabla_clase_split = pd.crosstab(df_combinado["class"], df_combinado["split"], normalize="columns").reindex(ORDEN_CLASES) * 100
display(tabla_clase_split.round(2))

Interseccion train/validation: 0 grupos
Interseccion train/test      : 0 grupos
Interseccion validation/test : 0 grupos

Particion sin fuga de grupos entre splits: True
Todas las imagenes tienen un split asignado.

Distribucion de clases por split (porcentaje dentro de cada split):


split,test,train,validation
class,,,
AK,3.86,3.43,2.93
BCC,13.31,13.01,13.41
BKL,10.61,10.57,9.14
DF,0.76,0.98,0.94
MEL,17.02,17.81,18.88
NV,51.23,50.70,51.02
SCC,2.34,2.48,2.62
VASC,0.87,1.02,1.05


## Ejemplo concreto: el grupo con mas imagenes

La verificacion anterior confirma de forma agregada (conteos) que no hay fuga de grupos entre splits, pero un ejemplo concreto hace tangible por que agrupar por lesion es necesario. Se identifica el grupo (lesion o imagen sin `lesion_id`) con mayor numero de imagenes en todo el dataset, y se confirma explicitamente que todas sus imagenes cayeron en el mismo split.

Si la particion se hubiera hecho por imagen individual en vez de por lesion, las imagenes de este grupo podrian haber quedado repartidas entre train y test, permitiendo que el modelo "memorice" esa lesion especifica en lugar de aprender el patron diagnostico general - precisamente el riesgo de fuga que esta particion evita.

In [7]:
tamano_grupo = df_combinado.groupby("group_id").size()
grupo_mas_grande = tamano_grupo.idxmax()
n_imagenes_grupo = int(tamano_grupo.max())

filas_grupo = df_combinado[df_combinado["group_id"] == grupo_mas_grande]
clase_grupo = filas_grupo["class"].iloc[0]
splits_presentes = filas_grupo["split"].unique()

print(f"Grupo con mas imagenes en el dataset: {grupo_mas_grande}")
print(f"Clase diagnostica: {clase_grupo}")
print(f"Numero de imagenes en este grupo: {n_imagenes_grupo}")
print(f"Splits en los que aparece este grupo: {list(splits_presentes)}")
print(f"Todas sus imagenes en el mismo split: {len(splits_presentes) == 1}")


Grupo con mas imagenes en el dataset: BCN_0001728
Clase diagnostica: MEL
Numero de imagenes en este grupo: 31
Splits en los que aparece este grupo: ['train']
Todas sus imagenes en el mismo split: True


## Persistencia de la particion

Se guarda el resultado final (una fila por imagen, con su clase, `group_id` y `split` asignado) en `reports/splits/isic2019/`, versionado en git. Este archivo sera la entrada unica de datos para los notebooks de construccion y entrenamiento de M1 y M4 (`03`, `04`, `05`), evitando repetir la logica de particionamiento y garantizando que ambos modelos se evaluen bajo exactamente la misma particion, tal como exige el protocolo comparativo.

In [8]:
df_particion_final = df_combinado[
    ["image", "class", "class_index", "group_id", "split", "image_path_relative"]
].copy()

PARTICION_CSV = SPLITS_DIR / "isic2019_particion_70_15_15_seed42.csv"
df_particion_final.to_csv(PARTICION_CSV, index=False)

print(f"Particion guardada en: {PARTICION_CSV}")
print(f"Filas guardadas: {len(df_particion_final)}")
display(df_particion_final.head())

print("\nResumen final de conteo por split:")
print(df_particion_final["split"].value_counts())

Particion guardada en: /home/milat/proyecto_de_titulo/reports/splits/isic2019/isic2019_particion_70_15_15_seed42.csv
Filas guardadas: 25331


,image,class,class_index,group_id,split,image_path_relative
0,ISIC_0000000,NV,5,ISIC_0000000,train,data/isic2019/ISIC_2019_Training_Input/ISIC_00...
1,ISIC_0000001,NV,5,ISIC_0000001,validation,data/isic2019/ISIC_2019_Training_Input/ISIC_00...
2,ISIC_0000002,MEL,4,ISIC_0000002,train,data/isic2019/ISIC_2019_Training_Input/ISIC_00...
3,ISIC_0000003,NV,5,ISIC_0000003,validation,data/isic2019/ISIC_2019_Training_Input/ISIC_00...
4,ISIC_0000004,MEL,4,ISIC_0000004,train,data/isic2019/ISIC_2019_Training_Input/ISIC_00...



Resumen final de conteo por split:
split
train         17705
validation     3818
test           3808
Name: count, dtype: int64


## Resumen y estado al cierre de este notebook

**Verificado con datos reales de ISIC 2019:**

- Entrada: `reports/inventory/isic2019_inventario_ocho_clases.csv` (generado en `01_datos_isic2019.ipynb`), cruzado con `ISIC_2019_Training_Metadata.csv` oficial.
- `lesion_id` disponible para 23.247/25.331 imagenes (11.847 lesiones unicas); 2.084 imagenes sin `lesion_id` tratadas como grupo unico.
- Verificado: 0 lesiones con clases diagnosticas inconsistentes entre sus propias imagenes.
- Particion construida a nivel de grupo (lesion o imagen individual), estratificada por clase, semilla 42.
- Proporciones finales a nivel de imagen: train 69,89% (17.705), validation 15,07% (3.818), test 15,03% (3.808).
- Verificado: 0 grupos compartidos entre train/validation/test (sin fuga de datos por lesion).
- Verificado: distribucion de clases consistente entre los tres splits.

**Artefacto guardado (versionable en `reports/`):**

- `reports/splits/isic2019_particion_70_15_15_seed42.csv` (via `reports/splits/isic2019/`)

**Explicitamente fuera de alcance de este notebook:** normalizacion/rescalado de pixeles, aumento de datos, calculo de pesos de clase (`w_c = N / (K * n_c)`, que segun `configs/pilot_m1_m4.yaml` se calcula solo sobre el split `train`), construccion de los modelos M1/M4 y entrenamiento. Estos pasos corresponden a los notebooks `03_modelo_efficientnet_b4.ipynb`, `04_modelo_cnn_referencia_m4.ipynb` y `05_entrenamiento_y_mof.ipynb`, que deben leer `isic2019_particion_70_15_15_seed42.csv` como unica fuente de particion en vez de reconstruirla.

**Limitacion declarada:** la separacion por lesion se aplico solo donde `lesion_id` estaba disponible; las 2.084 imagenes sin `lesion_id` se trataron como lesiones independientes de una sola imagen, siguiendo el criterio mas conservador posible con la informacion disponible, pero sin poder descartar por completo que alguna de ellas comparta paciente/lesion con otra imagen del dataset.